    TF-IDF, keyword extraction, metadata features

In [1]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path("../").resolve()
sys.path.append(str(PROJECT_ROOT))

from src.config import (
    CLEANED_TRAIN_PATH,
    CLEANED_TEST_PATH,
    FINAL_TRAIN_SVM_PATH,
    FINAL_TEST_SVM_PATH,
    TRAIN_LABEL_PATH
)

from src.utils.utils import load_csv, save_csv

from src.features.tfidf_features import TFIDFFeatureExtractor
from src.features.metadata_features import MetadataFeatureExtractor
from src.features.feature_union import FeatureUnion

print("✔ Imports loaded")

✔ Imports loaded


In [2]:
train_df = load_csv(CLEANED_TRAIN_PATH)
test_df = load_csv(CLEANED_TEST_PATH)

print(train_df.shape, test_df.shape)
train_df.head()

(510, 8) (86, 7)


,title,venue,year,authors,doi,Label,id,abstract
0,tabled clp for reasoning over stream data,iclp,2016,"edmond jajaga, lule ahmedi",10.1109/icsc.2017.64,1,299,semantic technologies have been extensively us...
1,xai-law: a logic programming tool for modeling...,iclp,2025,"agostino dovier, talissa dreossi, andrea formi...",10.4204/eptcs.439.28,5,92,we propose an approach to model articles of th...
2,workshop proceedings of the 40th international...,iclp,2024,"esteban guerrero, juan carlos nieves",10.1007/978-3-031-74209-5_21,1,311,traditionally in the argumentation theory lite...
3,probabilistic active goal recognition,kr,2025,"chenyuan zhang, cristian rojas cardenas, hamid...",10.24963/kr.2025/85,5,136,in multi-agent environments effective interact...
4,heuristic strategies for accelerating multi-ag...,kr,2024,"biqing fang, fangzhen lin",10.24963/kr.2024/32,1,41,multi-agent epistemic planning mep is about ac...


In [3]:
tfidf = TFIDFFeatureExtractor(
    max_title_features=3000,
    max_abstract_features=8000,
    max_author_features=1500
)

metadata = MetadataFeatureExtractor()
union = FeatureUnion()

print("✔ Feature extractors initialized")

✔ Feature extractors initialized


In [4]:
X_train, train_meta_df = union.fit_transform(tfidf, metadata, train_df)

y_train = train_df["Label"].values

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)

X_train shape: (510, 7131)
y_train shape: (510,)


In [5]:
from sklearn.model_selection import train_test_split
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report

X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train,
    test_size=0.2,
    random_state=42,
    stratify=y_train
)

model = LinearSVC(C=1.0)
model.fit(X_tr, y_tr)

pred = model.predict(X_val)

print(classification_report(y_val, pred))

              precision    recall  f1-score   support

           1       0.49      0.65      0.56        26
           2       0.25      0.29      0.27        21
           3       0.00      0.00      0.00        17
           4       0.09      0.06      0.07        18
           5       0.33      0.35      0.34        20

    accuracy                           0.30       102
   macro avg       0.23      0.27      0.25       102
weighted avg       0.26      0.30      0.28       102



/Users/nhatnam/Documents/DM_252/Assignment/venv/lib/python3.13/site-packages/sklearn/svm/_base.py:1258: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


In [6]:
X_test = union.transform(tfidf, metadata, test_df)

print("X_test shape:", X_test.shape)

X_test shape: (86, 7131)


In [7]:
from scipy import sparse
import os
import pandas as pd

# ensure output dir exists
os.makedirs(FINAL_TRAIN_SVM_PATH.parent, exist_ok=True)
os.makedirs(FINAL_TEST_SVM_PATH.parent, exist_ok=True)

# =========================
# SAVE SPARSE FEATURES
# =========================
train_npz_path = FINAL_TRAIN_SVM_PATH.with_suffix(".npz")
test_npz_path = FINAL_TEST_SVM_PATH.with_suffix(".npz")

sparse.save_npz(train_npz_path, X_train)
sparse.save_npz(test_npz_path, X_test)

# =========================
# SAVE LABELS (FIXED)
# =========================

pd.Series(y_train).to_csv(TRAIN_LABEL_PATH, index=False)

print("✔ Features saved to data/final/")
print("Train features:", train_npz_path)
print("Test features:", test_npz_path)
print("Labels:", TRAIN_LABEL_PATH)

✔ Features saved to data/final/
Train features: /Users/nhatnam/Documents/DM_252/Assignment/data/processed/train_SVM_features.npz
Test features: /Users/nhatnam/Documents/DM_252/Assignment/data/processed/test_SVM_features.npz
Labels: /Users/nhatnam/Documents/DM_252/Assignment/data/processed/train_labels.csv
